A code and the respective language will be given. Will explain that with user's expertise (their chosen language).

Inputs - Code, language, user's expertise
Outputs - Input Code Explanation with user's expertise

In [ ]:
# imports and initialization

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import Markdown, display

load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')


if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

In [ ]:
# model setup
groq_url = "https://api.groq.com/openai/v1"
openrouter_url = "https://openrouter.ai/api/v1"

groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

models = ["inclusionai/ling-3.0-flash-fin:free", "openai/gpt-oss-120b", ]
clients = {"openai/gpt-oss-120b": groq, "inclusionai/ling-3.0-flash-fin:free": openrouter}

In [ ]:
# system and user prompt
# Constructs the user prompt string formatted with code language, code snippet, and user preferred language
def get_user_prompt(code_language, code, user_pref_language):
    user_prompt = f""" 
    Explain the given code {code} in {code_language} to the user. User is an expertise in {user_pref_language}, so explain with respect to their preferred language 
    """
    return user_prompt

system_prompt = """ You are a crtical thinker and an expert who explain what the code does, why it does and how it does. But keep the content crisp and small so that the user will not go away for another explanation"""

In [ ]:
#code to test for sample
code = """ #include <iostream>
#include <vector>
#include <string>

std::string longestCommonPrefix(const std::vector<std::string>& strs) {
    if (strs.empty()) return "";

    // Iterate through characters of the first string
    for (size_t i = 0; i < strs[0].size(); ++i) {
        char ch = strs[0][i];
        
        // Compare with the character at the same position in all other strings
        for (size_t j = 1; j < strs.size(); ++j) {
            if (i >= strs[j].size() || strs[j][i] != ch) {
                return strs[0].substr(0, i);
            }
        }
    }

    return strs[0];
}  """

In [ ]:
# main code that calls the openrouter api with mentioned chat parameters
# Calls the LLM model to generate an explanation of the given code tailored to the user preferred language
def explain_code(code_language, code, user_pref_language):
    response = openrouter.chat.completions.create(
        model=models[0], 
        messages=[
        {"role":"system", "content":system_prompt}, 
        {"role":"user", "content":get_user_prompt(code_language, code, user_pref_language)}])
    return response.choices[0].message.content

In [ ]:
# gradio ui - with required inputs and outputs
with gr.Blocks() as ui:
    with gr.Row():
        with gr.Column():
            code_lan = gr.Textbox(label="Enter Code language")
            user_pref = gr.Textbox(label="Enter Preferred language")
            cpp = gr.Code(
                label="Enter the code  ",
                lines = 25
            )
        with gr.Column():
            exp = gr.Markdown(label="Explanation") 
    with gr.Row():
        btn = gr.Button("Explain the code")
    btn.click(explain_code,inputs=[code_lan, cpp, user_pref], outputs=[exp] )
        
ui.launch()